# FG-TRACER: encoder-divergence probe
Bounds the vision-encoder confound. Gemma from attached Kaggle model; MedGemma
from HF (`google/medgemma-4b-it`, HF_TOKEN secret). Short run. Output:
`figures_compare_fixed/encoder_probe.json`.

In [ ]:
# 1. Deps. Single clean pip pass (no destructive uninstall, no --no-deps numpy
#    hack -- those left transformers half-installed so GenerationMixin failed to
#    import). Pin transformers to the 4.56 line (requirements: >=4.50,<5) and let
#    pip resolve the rest in ONE pass. Fail loudly on the exact imports we need.
%pip install -q -U "transformers>=4.56,<4.57" "huggingface_hub>=0.34" accelerate "datasets>=2.19" pillow safetensors
import transformers, torch
from transformers import AutoModelForImageTextToText
from transformers.generation import GenerationMixin
print('deps OK | transformers', transformers.__version__,
      '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())


In [ ]:
# extract/copy FG-TRACER code into the working dir (handles zip OR loose files)
import glob, os, shutil, zipfile
WORK='/kaggle/working'
zips=glob.glob('/kaggle/input/**/fgtracer_code.zip',recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    src=glob.glob('/kaggle/input/**/run_tracing.py',recursive=True)
    assert src,'attach the fgtracer-code dataset'
    cd=os.path.dirname(src[0])
    for it in os.listdir(cd):
        s=os.path.join(cd,it); d=os.path.join(WORK,it)
        if os.path.isdir(s): shutil.copytree(s,d,dirs_exist_ok=True)
        elif not os.path.exists(d): shutil.copy2(s,d)
os.chdir(WORK)
import fgtracer  # fail loudly if the package did not land
print('code ready:',sorted(os.listdir(WORK))[:12])

In [ ]:
# HF token loaded from private Kaggle dataset (not hardcoded in source).
# W&B key still from Kaggle Secrets (it's only used for logging, not model download).
import os, pathlib
HF_TOKEN = pathlib.Path('/kaggle/input/fgtracer-hf-token/hf_token.txt').read_text().strip()
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
# W&B: try Kaggle secret; if unavailable, skip logging gracefully
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret('WANDB_API_KEY')
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('HF token loaded | W&B authenticated')
except Exception as _e:
    print(f'HF token loaded | W&B skipped ({_e})')

MEDGEMMA = 'google/medgemma-4b-it'


In [ ]:
import glob, os
# Gemma: attached Kaggle model
cand=[os.path.dirname(c) for c in glob.glob('/kaggle/input/**/config.json',recursive=True)
      if 'gemma-3' in os.path.dirname(c).lower() and 'medgemma' not in os.path.dirname(c).lower()]
assert cand,'attach google/gemma-3 transformers model'
GEMMA=sorted(cand,key=len)[0]
# MedGemma: HuggingFace

In [ ]:
# Deep XAI (GPU-cheap): one instrumented forward/sample per model -> logit-lens
# trajectory, lens-DLA, entropy, attention-to-image, PLUS encoder-divergence
# (CKA/cosine) confound bound and cross-model KL/JS. Each model loads once.
# Deep-XAI calls are non-fatal; if the confound bound wasn't produced we fall
# back to encoder_probe.py so the reviewer-critical CKA is guaranteed.
import subprocess, json, os
OUT='figures_compare_fixed/xai'
def run(cmd):
    try: subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e: print('WARN non-fatal:', e)
run(['python','analysis/deep_xai.py','--model_dir',GEMMA,'--label','gemma','--out_dir',OUT,'--num_samples','100','--hf_split','test'])
run(['python','analysis/deep_xai.py','--model_dir',MEDGEMMA,'--label','medgemma','--out_dir',OUT,'--num_samples','100','--hf_split','test'])
run(['python','analysis/deep_xai.py','--combine','--out_dir',OUT])
# guarantee the confound bound even if deep_xai had trouble
if not os.path.exists('figures_compare_fixed/encoder_probe.json'):
    print('deep_xai did not emit encoder_probe.json -> fallback encoder_probe.py')
    subprocess.run(['python','analysis/encoder_probe.py','--base_dir',GEMMA,'--new_dir',MEDGEMMA,
                    '--num_samples','100','--hf_split','test','--out','figures_compare_fixed/encoder_probe.json'],check=True)
for f in ['encoder_probe.json','xai/xai_divergence.json','xai/xai_gemma.json','xai/xai_medgemma.json']:
    p=os.path.join('figures_compare_fixed',f)
    if os.path.exists(p):
        d=json.load(open(p)); d.pop('per_sample',None)
        print('===',f,'==='); print(json.dumps(d,indent=1)[:800])


In [ ]:
# W&B cloud logging is optional: a missing or rejected key must not fail the run
import json, os, glob
try:
    import wandb
    run = wandb.init(project="fgtracer", name="fgtracer-encoder-kaggle",
                     tags=["encoder", "cka", "deep_xai", "kaggle"],
                     config={"n_samples": 100, "split": "vqa_rad"})
    # log encoder confound verdict
    probe_path = "figures_compare_fixed/encoder_probe.json"
    if os.path.exists(probe_path):
        d = json.load(open(probe_path))
        cka = d.get("linear_cka", 0) or 0
        verdict = "STRONG" if cka >= 0.9 else "MODERATE" if cka >= 0.75 else "WEAK"
        wandb.log({"encoder/linear_cka": cka,
                   "encoder/cosine_similarity": d.get("cosine_similarity_mean", 0),
                   "encoder/verdict": verdict})
        print(f"CKA={cka:.4f} verdict={verdict}")
    # upload xai outputs
    xai_dir = "figures_compare_fixed/xai"
    if os.path.exists(xai_dir):
        artifact = wandb.Artifact("outputs-encoder-xai", type="results", description="Deep XAI + encoder probe outputs")
        artifact.add_dir(xai_dir)
        run.log_artifact(artifact)
    wandb.finish()
    print("W&B run logged:", run.url)
except Exception as _e:
    print(f'W&B logging skipped ({_e})')
